# Insulin response predictor - guided run

This notebook runs the same tested pipeline as the command line and displays its results
inline, with plain-language explanations alongside every step. No statistics or machine
learning background is required to follow it. **Research only: no output in this notebook
is a treatment instruction.**


## What this notebook does

- Checks whether the exported information is complete and internally consistent.
- Groups meals, insulin, and glucose readings into comparable historical episodes.
- Tests whether later glucose can be predicted better than simple baselines.
- Stops if the prediction is not sufficiently reliable or behaves implausibly.
- If the checks pass, compares several retrospective candidate-dose methods.

## What this notebook does not do

- It does not provide a dose for the next meal.
- It does not replace a clinician or an established insulin calculator.
- It does not establish that changing a historical dose would have caused a particular
  result.
- A PASS means only that further retrospective analysis is permitted.

## The five steps, in plain language

1. **Settings** - choose fictional demo data or your own local CSV exports.
2. **Load and validate** - the data is read in and checked for completeness.
3. **Run the pipeline** - the same tested code the command line uses, run once, start to
   finish.
4. **Review results** - each report and chart is shown with an explanation of what it
   means and why it may stop early.
5. **Summary and next steps** - a short recap of what happened and what to do next.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / 'pyproject.toml').exists() else Path.cwd()
%pip install -q -e "$PROJECT_ROOT"


## 1. Settings

Leave `MODE` as `'fictional'` for the complete, safe demo. Change it to `'csv'` only after
placing the four sheet exports in an ignored local directory and creating
`config/subject.yaml`.

| Setting | Meaning | What most users should choose |
|---|---|---|
| `MODE` | Whether to use fictional or local CSV data | Start with `'fictional'` |
| `INPUT_DIRECTORY` | Folder containing exported CSV files | Leave unchanged for the demo |
| `OUTPUT_DIRECTORY` | Folder where reports will be written | Usually leave unchanged |
| `SUBJECT_CONFIG` | Local research settings | Required only for CSV mode |
| `FICTIONAL_DAYS` | Size of the fictional demonstration | Leave at 90 |
| `FICTIONAL_SEED` | Makes the demo repeatable | Leave unchanged |

- **Fictional mode** contains no personal information; it is entirely made up for
  demonstration purposes.
- **CSV mode** reads local files from `INPUT_DIRECTORY` on your own computer.
- Neither `data/raw` nor `notebook-output` should ever be committed to version control -
  both are already excluded by `.gitignore`.


In [ ]:
MODE = 'fictional'  # 'fictional' or 'csv'
INPUT_DIRECTORY = PROJECT_ROOT / 'data' / 'raw' / 'latest'
OUTPUT_DIRECTORY = PROJECT_ROOT / 'notebook-output'
SUBJECT_CONFIG = PROJECT_ROOT / 'config' / 'subject.yaml'
FICTIONAL_DAYS = 90
FICTIONAL_SEED = 42


## 2. Load data and validate

### What information is required

| File | What it contains | Mandatory? |
|---|---|---|
| `glucose.csv` | Glucose readings and when they occurred | Yes |
| `food.csv` | Meal description, carbohydrate estimate, meal type, and optional protein/fat | Yes |
| `insulin.csv` | Dose amount, insulin type, timing, and the reason for the dose | Yes |
| `context.csv` | Exercise, illness, stress, poor sleep, alcohol, or similar factors | Optional |
| `meal_references.csv` | Reusable details for meals eaten repeatedly | Optional |

In plain terms:

- **Glucose** - readings and when they occurred.
- **Food** - meal description, carbohydrate estimate, and optional nutritional details.
- **Insulin** - amount, type, timing, and reason.
- **Context** - exercise, illness, stress, poor sleep, alcohol, or similar factors.

`meal_references.csv` represents meals eaten repeatedly. A `meal_reference_id` in
`food.csv` can reuse standard food details from that lookup, saving re-entry for the same
meal. Any explicit value already present in `food.csv` overrides the reference value - the
lookup only fills in blanks.

The cell below loads whichever data source `MODE` selected and checks it against these
requirements before anything else runs.


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from insulin_response_predictor.configuration import load_policy_config
from insulin_response_predictor.io import load_csv_exports
from insulin_response_predictor.pipeline import run_pipeline
from insulin_response_predictor.policy import PolicyConfig
from insulin_response_predictor.synthetic import generate_synthetic_dataset
from insulin_response_predictor.validation import validate_dataset

if MODE == 'fictional':
    tables = generate_synthetic_dataset(
        days=FICTIONAL_DAYS, seed=FICTIONAL_SEED, scenario='identifiable'
    )
    policy_config = PolicyConfig()
elif MODE == 'csv':
    tables = load_csv_exports(INPUT_DIRECTORY)
    policy_config = load_policy_config(SUBJECT_CONFIG)
else:
    raise ValueError("MODE must be 'fictional' or 'csv'")

issues = validate_dataset(tables)
issue_table = pd.DataFrame([issue.__dict__ for issue in issues])
display(issue_table if not issue_table.empty else Markdown('**Validation passed with no issues.**'))
if any(issue.severity == 'error' for issue in issues):
    raise ValueError('Fix validation errors before continuing.')


### How to read this result

- **Error** rows describe a problem that makes a record unusable or unsafe to model -
  modelling must not continue while errors remain, and the cell above stops the notebook
  if any are present.
- **Warning** rows describe something worth checking but that does not block the run.

Three example problems this check can catch:

- A timestamp with a **missing timezone**, which makes it impossible to place an event in
  time relative to others.
- An **impossible glucose or carbohydrate value** (for example, a negative number or one
  far outside any plausible human range).
- An **unknown meal or dose category** that does not match one of the accepted values (for
  example, a `meal_type` that is not `breakfast`, `lunch`, `dinner`, `snack`, or
  `hypo_treatment`).

If you see errors, fix the source spreadsheet and re-export it rather than editing the
generated report or this notebook's output - the report is only as trustworthy as the data
that produced it.


## 3. Run the complete pipeline

This single call runs the entire tested pipeline: it builds episodes, runs exploratory
analysis, evaluates forward models against the statistical gate, and - only if that gate
passes - runs the retrospective policy comparisons. Nothing about the gate is manual; it is
evaluated automatically below and cannot be skipped.


In [ ]:
manifest = run_pipeline(tables, OUTPUT_DIRECTORY, policy_config=policy_config)
display(pd.DataFrame(manifest['stages']).T)
print('Final status:', manifest['status'])


## 4. Review reports and charts

The rest of the notebook walks through every report and chart the pipeline produced, in
the order the pipeline created them. Each result is introduced with a plain-language
explanation before it is shown.


### 4.1 Data quality report

This is the same completeness and consistency check explained above, now shown as the full
report the pipeline saved to disk, including a three-panel timeline of every event.


In [ ]:
data_quality_report = OUTPUT_DIRECTORY / '01_data_quality' / 'data_quality.md'
if data_quality_report.exists():
    display(Markdown(data_quality_report.read_text(encoding='utf-8')))


### 4.2 Episodes and exploratory analysis

#### What is an episode?

A usable episode connects four things:

1. A meal.
2. Its associated rapid insulin dose.
3. A recent glucose reading before the meal.
4. A glucose result approximately two to four hours later.

Every episode is labelled with a status:

- **Clean** - suitable for primary modelling. Nothing else interfered between the meal and
  the outcome reading.
- **Contaminated** - a usable outcome exists, but another meal, a hypo treatment, or
  another rapid dose happened in between, so the cause of the outcome cannot be isolated.
- **Unusable** - required information (a bolus, a pre-meal reading, or an outcome reading)
  is missing.

Only **clean** episodes enter the models below, because only they isolate the relationship
between a single meal, a single dose, and the outcome that followed.

#### What exploratory analysis shows

Exploratory analysis looks at the clean episodes before any model is trained, to get a
first plain read on the data:

- The **meal-response chart** shows how glucose changed after meals across the clean
  episodes, so you can see the range of outcomes before any model tries to predict them.
- The **carbohydrate-response chart** shows how the glucose change relates to meal size, to
  give a first-pass sense of whether bigger meals produced bigger swings.
- The **comparable-meal repeatability estimate** asks whether broadly similar historical
  meals produced similar outcomes. Large differences suggest that important factors may be
  missing from the recorded data, or that the response is naturally variable from meal to
  meal - it does not, by itself, prove either explanation.

<details>
<summary><strong>Technical note</strong>: what "comparable" means here</summary>

Comparable meals are matched by meal type, similar carbohydrate amount, and similar
pre-meal glucose. This is a descriptive spread statistic over matched historical rows, not
a formal statistical error term and not evidence that any specific factor caused the
outcome.
</details>


In [ ]:
eda_report = OUTPUT_DIRECTORY / '02_exploratory_analysis' / 'eda_report.md'
if eda_report.exists():
    display(Markdown(eda_report.read_text(encoding='utf-8')))

meal_response_chart = OUTPUT_DIRECTORY / '02_exploratory_analysis' / 'meal_response.png'
if meal_response_chart.exists():
    from IPython.display import Image

    display(Image(filename=str(meal_response_chart)))


### 4.3 Forward-model results

#### What the models are trying to do

Each model predicts the *later glucose observation* for a clean episode - it does not
predict, and is not used to choose, an insulin dose. The two simplest models are baselines:
a more complex model is only useful if it consistently beats them.

| Model | Plain-language description |
|---|---|
| Persistence | Assumes glucose will remain near its starting value |
| Linear extrapolation | Extends the latest visible glucose trend |
| Ridge | Learns a stable weighted relationship between the recorded factors |
| Histogram gradient boosting | Learns nonlinear combinations and thresholds |
| Random Forest | Combines many decision trees |

#### Glossary of the numbers below

- **RMSE** (root mean squared error) - typical prediction error, with more weight on large
  misses; lower is better.
- **MAE** (mean absolute error) - average absolute miss; lower is better.
- **Skill versus persistence** - improvement over assuming no glucose change; above zero is
  better.
- **Directional accuracy** - how often the model correctly predicts whether glucose rises
  or falls.
- **Confidence interval** - the range of results seen when the test episodes are repeatedly
  resampled; a wide range means the estimate is less certain.
- **Rolling-origin evaluation** - repeatedly trains on earlier history and tests on later
  history, to check the result is not a one-off.
- **Dose sensitivity** - checks whether increasing recorded insulin generally moves the
  model's prediction downward, the way it should physiologically.

<details>
<summary><strong>Technical note</strong>: what feeds the models</summary>

Besides the pre-meal glucose, carbohydrate, and meal type shown above, each model also uses
rapid **insulin on board (IOB)** - the estimated fraction of a previous rapid dose still
active - and **carbohydrate on board (COB)** - the estimated fraction of a previous meal
still being absorbed - along with time of day, recent corrections, and recent
hypoglycaemia and exercise. These are computed strictly from information available at the
time of the bolus, so no future information leaks into a prediction.
</details>


In [ ]:
forward_report = OUTPUT_DIRECTORY / '03_forward_model' / 'forward_report.md'
if forward_report.exists():
    display(Markdown(forward_report.read_text(encoding='utf-8')))


### 4.4 The forward-model gate: PASS or STOP

**PASS** means:

- At least one learned model beat persistence by the configured margin.
- Its insulin-response direction was plausible within historical support.
- Retrospective policy comparisons may run.
- It is **not** approval for real-world dosing.

**STOP** means:

- The information does not support a sufficiently reliable forward model.
- Policy analysis is deliberately skipped.
- STOP is a valid and useful result.
- Recommended next actions: collect more episodes, improve recording consistency, and
  review for missing variables.

The cell below reports which one happened for this run, computed directly from the
pipeline's own manifest.


In [ ]:
forward_stage = manifest['stages']['forward_model']
if forward_stage.get('any_model_passes'):
    gate_message = (
        "#### PASS - a forward model cleared the statistical gate\n\n"
        f"Best-performing model: **{forward_stage.get('best_model')}**. "
        "Retrospective policy comparisons will run below. "
        "This is not approval for real-world dosing."
    )
else:
    gate_message = (
        "#### STOP - no forward model cleared the statistical gate\n\n"
        "The policy experiment is deliberately skipped below. STOP is a valid, useful "
        "research result - see the summary and troubleshooting sections at the end for "
        "suggested next steps."
    )
display(Markdown(gate_message))


### 4.5 Diagnostic charts

- Points near the diagonal line indicate closer predictions.
- Residuals (actual minus predicted) should be scattered around zero with no obvious
  pattern.
- A pattern by time of day may indicate a missing time-related effect.
- A pattern by meal size may indicate a problem with carbohydrate estimates or with the
  model's structure.
- A good-looking chart alone is not sufficient validation - it is one check among the
  metrics and the gate above, not a replacement for them.


In [ ]:
for chart in [
    OUTPUT_DIRECTORY / '03_forward_model' / 'predicted_vs_actual.png',
    OUTPUT_DIRECTORY / '03_forward_model' / 'residual_diagnostics.png',
]:
    if chart.exists():
        from IPython.display import Image

        display(Markdown(f'#### {chart.stem.replace("_", " ").title()}'))
        display(Image(filename=str(chart)))


### 4.6 Retrospective policy comparison

This section only runs when the gate above is PASS. It compares four ways of arriving at a
retrospective candidate dose for each historical clean episode - none of them are
recommendations.

| Policy | Meaning |
|---|---|
| Configured formula | Uses the locally configured ICR and ISF |
| Fitted formula | Estimates a formula from historically in-range episodes |
| Historical imitation | Tries to reproduce previous dosing patterns |
| Model inversion | Searches supported historical dose values for a model-predicted target |

**ICR** (insulin-to-carbohydrate ratio) and **ISF** (insulin sensitivity factor) are the
two numbers the configured and fitted formulas are built from.

- **Historical imitation** means "similar to past behaviour," not "optimal."
- **Model inversion** is scored by the same model used to select it, so its result is
  optimistic by construction.
- Every candidate is bounded by historical, meal-specific dose support and rounded to the
  nearest 0.5 unit.
- An **abstention** means the episode fell outside the supported conditions, so no
  candidate is produced for it.
- None of these values are recommended, prescribed, safe, or optimal doses - they are
  retrospective research candidates only.

<details>
<summary><strong>Technical note</strong>: how model inversion searches for a dose</summary>

Model inversion evaluates the selected forward model across every historical dose value
supported for that meal type, in 0.5-unit steps, and keeps the value whose predicted
outcome lands closest to the target range (with an extra penalty for predictions that land
below the low threshold). It never proposes a dose outside what was actually observed for
that meal type.
</details>


In [ ]:
policy_report = OUTPUT_DIRECTORY / '04_policy_experiment' / 'policy_report.md'
if policy_report.exists():
    display(Markdown(policy_report.read_text(encoding='utf-8')))
else:
    display(Markdown(
        '_No policy report was generated for this run. This is expected whenever the '
        'forward-model gate result above is STOP._'
    ))


## 5. Summary and next steps

A short recap of this run, generated automatically from the pipeline's own manifest, and a
suggested next action based on what happened.


In [ ]:
stages = manifest['stages']
eda_stage = stages.get('exploratory_analysis', {})
forward_stage = stages.get('forward_model', {})
policy_stage = stages.get('policy_experiment', {})

if manifest['status'] == 'validation_failed':
    next_action = 'Correct the source records flagged above, then re-export and rerun.'
elif manifest['status'] == 'stopped_at_forward_gate':
    next_action = (
        'Improve recording coverage and consistency and collect more clean episodes, '
        'then rerun.'
    )
elif manifest['status'] == 'complete' and MODE == 'fictional':
    next_action = 'The software demonstration succeeded; prepare your own CSV exports next.'
else:
    next_action = (
        'Review this report with a qualified clinician or researcher before deciding '
        'whether further analysis is justified.'
    )

summary = pd.Series(
    {
        'Pipeline status': manifest['status'],
        'Clean episodes': eda_stage.get('clean_episodes', 'n/a'),
        'Best forward model': forward_stage.get('best_model', 'n/a'),
        'Forward gate passed': forward_stage.get('any_model_passes', False),
        'Policy evaluation ran': policy_stage.get('status') == 'complete',
        'Output location': str(OUTPUT_DIRECTORY),
        'Suggested next action': next_action,
    }
)
display(summary.to_frame(name='Value'))


### Interpretation boundary

A PASS means only that retrospective experimentation met the configured statistical gate.
It does not validate a dose, make a prescription, or establish safety for real-world use.


## 6. Troubleshooting

- **Python kernel not selected** - choose the project's virtual environment as the
  notebook kernel (Kernel menu, or the kernel picker in your editor) before running any
  cell.
- **Package import failure** - restart the kernel and re-run the first cell, which installs
  the project in editable mode; confirm you are running inside the project's own virtual
  environment (`.venv`).
- **Missing CSV file** - confirm `glucose.csv`, `food.csv`, and `insulin.csv` exist in
  `INPUT_DIRECTORY` using exactly those filenames; `context.csv` is optional.
- **Missing `subject.yaml`** - copy `config/subject.example.yaml` to the git-ignored
  `config/subject.yaml` and fill in your own values; this is required only in CSV mode.
- **Validation errors** - fix the source spreadsheet and re-export; do not edit the
  generated report or this notebook's output directly.
- **No clean episodes** - check that rapid boluses are recorded within about 30 minutes of
  a meal and typed as a meal dose, and that pre-meal and outcome glucose readings exist
  around each one; more consistent logging over time usually resolves this.
- **Forward gate STOP** - a valid research result, not an error; see the summary above for
  suggested next steps.
- **Policy folder not created** - expected whenever the gate result is STOP; the policy
  stage is intentionally skipped, not broken.
